# 🚀 Kronos-base Cloud GPU Inference Server
Run Kronos foundation model inference on Google Colab GPU (T4 / A100) and connect directly to your Laravel application.

### Instructions:
1. **Runtime Type**: Go to `Runtime` > `Change runtime type` > select **T4 GPU** (free).
2. Run **Step 1** to confirm GPU availability.
3. Run **Step 2** to clone official Kronos architecture and your custom GitHub repository.
4. Run **Step 3** to install requirements.
5. Run **Step 4** to start the FastAPI server with an **ngrok** tunnel and get your `KRONOS_API_URL`.

In [ ]:
# Step 1: Verify Colab GPU acceleration
!nvidia-smi

In [ ]:
# Step 2: Clone Kronos architecture and your custom microservice repository
# 2a. Official Kronos model code
!rm -rf /content/Kronos
!git clone https://github.com/shiyu-coder/Kronos.git /content/Kronos

# 2b. Clone your standalone Kronos microservice repository
# REPLACE with your new GitHub repo URL if you created a dedicated one:
YOUR_GITHUB_REPO = "https://github.com/janakkafle/kronos-service.git"

!rm -rf /content/kronos_service
!git clone $YOUR_GITHUB_REPO /content/kronos_service

# If you haven't pushed to a dedicated repo yet, you can also upload files directly or clone the subfolder.

In [ ]:
# Step 3: Install all required Python packages
!pip install -q fastapi uvicorn pydantic pyngrok transformers accelerate safetensors pandas numpy requests

In [ ]:
# Step 4: Start PyNgrok tunnel and launch FastAPI inference server
import sys
import os

# Add Kronos official repository & microservice to Python path
sys.path.append('/content/Kronos')
sys.path.append('/content/kronos_service')

# (Optional) Set your ngrok authtoken if you have an account (free at https://dashboard.ngrok.com/get-started/your-authtoken)
NGROK_AUTH_TOKEN = ""
from pyngrok import ngrok
if NGROK_AUTH_TOKEN:
    ngrok.set_auth_token(NGROK_AUTH_TOKEN)

# Start tunnel on port 8000
ngrok.kill()
tunnel = ngrok.connect(8000)
public_url = tunnel.public_url

print("\n" + "="*60)
print("⚡ KRONOS GPU INFERENCE SERVER IS ONLINE!")
print(f"Public Tunnel URL: {public_url}")
print("="*60)
print(f"\nCopy & paste this into your Laravel .env file:")
print(f"KRONOS_API_URL={public_url}")
print("="*60 + "\n")

# Launch FastAPI server (this keeps the cell running and serving requests)
%cd /content/kronos_service
!uvicorn app:app --host 0.0.0.0 --port 8000